# ⚽ Predict the FIFA World Cup 2026

## 📖 Background

The 2026 FIFA World Cup is one of the biggest sporting events in the world, hosted across the United States, Canada, and Mexico. For the first time, the tournament expands to 48 teams, producing 104 matches across the group stage and knockout rounds.

Using machine learning, historical statistics, and soccer domain knowledge, predict match scores, corners, and cards for every fixture. You must submit all your predictions before a single ball is kicked.

The scoring system rewards precision: an exact scoreline earns maximum points, while close predictions still earn partial credit. Later rounds carry score multipliers, so a strong model that holds up in the knockout stages can leapfrog the competition. The challenge is designed to be difficult enough that no one can achieve a perfect score—even with AI assistance—but accessible enough that any data enthusiast can participate and score points.

## 💾 The data

You have access to the following files:

#### `data/group_fixtures.csv` — all 72 group stage matches
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `group` | Group letter (A–L) |
| `home_team` | Home team name |
| `away_team` | Away team name |
| `date` | Match date (UTC) |
| `venue` | Stadium and city |

#### `data/knockout_slots.csv` — all 32 knockout round slots
| Variable | Description |
|---|---|
| `match_id` | Unique match identifier |
| `round` | Round name (e.g. `Quarter-final`) |
| `multiplier` | Score multiplier for this round |
| `slot_home` | Description of the home team slot (e.g. `Winner Group A`) |
| `slot_away` | Description of the away team slot |

| Variable | Description |
|---|---|

You may also bring in any external data—FIFA rankings, historical match results, player statistics—to build your predictions.

In [41]:
import pandas as pd

group_fixtures = pd.read_csv('data/group_fixtures.csv')
group_fixtures.head()

,match_id,group,home_team,away_team,date_utc,venue
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City"
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara"
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto"
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles"
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver"


In [42]:
knockout_slots = pd.read_csv('data/knockout_slots.csv')
knockout_slots

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F)
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H)
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I)
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K)
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J)
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J)


## 💪 Competition challenge

The 2026 World Cup has two phases:

- **Group stage** (matches 1–72): The 48 teams are split into 12 groups of 4. Every team plays the other 3 teams in their group once. The best teams from each group advance to the next phase.
- **Knockout stage** (matches 73–104): Single-elimination rounds — Round of 32, Round of 16, Quarter-finals, Semi-finals, and the Final. Lose once and you're out. Crucially, the two teams playing in each knockout match are not known in advance: they depend on who qualified from the group stage.

Submit predictions for **every match** in both phases. For each match you need to predict:

1. **Score** — the exact final scoreline (e.g. `2-1` means the home team scores 2, the away team scores 1). For knockout matches, the score is the result after 90 minutes and extra time — the penalty shootout is not included.
2. **Corners** — the number of corner kicks awarded in the match
3. **Yellow cards** — the number of yellow cards shown in the match
4. **Red cards** — the number of red cards shown in the match

For **group stage** matches, also predict:
- **Winning team** — which team wins the individual match (use `home`, `away`, or `draw`)

For **knockout round** matches, also predict:
- **Matchup** — which two teams you predict will be playing in that slot. Because the bracket is determined by group stage results, you need to predict which teams advance far enough to meet in each round.
- **Match winner** — which team wins the match (use `home` or `away`)
- **Penalties** — whether the match goes to a penalty shootout (`True` or `False`)

### Scoring system

| Category | Condition | Points |
|---|---|---|
| Score | Exact scoreline | 25 |
| Score | Correct goal difference, wrong score | 10 |
| Score | Correct total goals, wrong score | 10 |
| Corners | Exact number | 10 |
| Corners | Off by 2 | 5 |
| Yellow cards | Exact number | 10 |
| Yellow cards | Off by 1 | 5 |
| Red cards | Exact number | 5 |
| Winning team *(group stage only)* | Correct | 40 |
| Matchup *(knockout only)* | Both teams correct | 20 |
| Matchup *(knockout only)* | One team correct | 10 |
| Match winner *(knockout only)* | Correct | 20 |
| Penalties *(knockout only)* | Correct | 5 |

All points for a match are multiplied by the round factor:

| Round | Multiplier |
|---|---|
| Group stage | ×1 |
| Round of 32 | ×1 |
| Round of 16 | ×2 |
| Quarter-final | ×4 |
| Semi-final | ×8 |
| Third-place playoff | ×8 |
| Final | ×16 |

## 🗓️ Group stage predictions

Fill in your predictions for all 72 group stage matches below.

In [43]:
group_predictions = group_fixtures.copy()

# Fill in your predictions for each match
# Example (match 1 — Mexico vs South Africa): predicted_home_goals=2, predicted_away_goals=1, corners=9, yellow_cards=3, red_cards=0, winning_team='home'
group_predictions['predicted_home_goals'] = None   # e.g. 2
group_predictions['predicted_away_goals'] = None   # e.g. 1
group_predictions['corners']              = None   # e.g. 9
group_predictions['yellow_cards']         = None   # e.g. 3
group_predictions['red_cards']            = None   # e.g. 0
group_predictions['winning_team']         = None   # "home", "away", or "draw"

group_predictions

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",None,None,None,None,None,None
1,2,A,South Korea,UEFA Playoff D,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",None,None,None,None,None,None
2,3,B,Canada,UEFA Playoff A,2026-06-12T19:00:00Z,"BMO Field, Toronto",None,None,None,None,None,None
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",None,None,None,None,None,None
4,5,D,Australia,UEFA Playoff C,2026-06-13T04:00:00Z,"BC Place, Vancouver",None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",None,None,None,None,None,None
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",None,None,None,None,None,None
69,70,K,FIFA Playoff 1,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",None,None,None,None,None,None
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",None,None,None,None,None,None


## 🏆 Knockout stage predictions

For knockout matches you also predict **which teams are playing**. Fill in the team names based on your group stage predictions, then add your match predictions.

In [44]:
knockout_predictions = knockout_slots.copy()

# Fill in your predictions for each knockout match
# Example (match 73 — Round of 32): predicted_home_team='Brazil', predicted_away_team='France', predicted_home_goals=1, predicted_away_goals=0, corners=8, yellow_cards=2, red_cards=0, match_winner='home', penalties=False
knockout_predictions['predicted_home_team']  = None   # e.g. "Brazil"
knockout_predictions['predicted_away_team']  = None   # e.g. "France"
knockout_predictions['predicted_home_goals'] = None   # e.g. 1
knockout_predictions['predicted_away_goals'] = None   # e.g. 0
knockout_predictions['corners']              = None   # e.g. 8
knockout_predictions['yellow_cards']         = None   # e.g. 2
knockout_predictions['red_cards']            = None   # e.g. 0
knockout_predictions['match_winner']         = None   # "home" or "away"
knockout_predictions['penalties']            = None   # True or False

knockout_predictions

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,None,None,None,None,None,None,None,None,None
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,None,None,None,None,None,None,None,None,None
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),None,None,None,None,None,None,None,None,None
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,None,None,None,None,None,None,None,None,None
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,None,None,None,None,None,None,None,None,None
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),None,None,None,None,None,None,None,None,None
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),None,None,None,None,None,None,None,None,None
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),None,None,None,None,None,None,None,None,None
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),None,None,None,None,None,None,None,None,None
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),None,None,None,None,None,None,None,None,None


## ✅ Checklist before publishing into the competition

- Rename your workspace to make it descriptive of your work. N.B. you should leave the notebook name as `notebook.ipynb`.
- Remove redundant cells like the judging criteria, so the workbook is focused on your predictions.
- Make sure all prediction cells are filled in—`None` values will score 0 points.
- Check that all cells run without error.
- Make sure your workbook is published before **June 10, 2026 at 09:00 UTC**.

## ⏳ Time is ticking. Good luck!

In [45]:
import numpy as np
import pandas as pd
import pickle
import copy
from collections import defaultdict
import random
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*feature names.*")
from feature_engineering import (
    resolve_team_original_to_updated, resolve_team_updated_to_original,
    combine_teams_elo, add_match_features, is_home_advantage,
    get_team_hist
    )
from final_pipeline import run_mc_pipeline, fill_predictions_df


In [46]:
elo_ratings = pd.read_csv('data/elo.csv', on_bad_lines="skip")

In [47]:
rawdf = pd.read_csv(r"data/raw_history.csv")
team_hist = get_team_hist(rawdf)

## **Match Simulation Setup**
- simulate_match
- group_simulate
- get_round_of_32
- knockout_simulate
- wc_simulate
- monte_carlo_simulate

In [48]:
def simulate_match(
    match_row, # this should contain date, home_team, away_team
    
    # =========================
    # MODELS (grouped)
    # =========================
    models,
    
    # =========================
    # DATA
    # =========================
    elo_ratings, # elo ratings
    team_hist, # team history last 5 - 10 matches
    
    # =========================
    # FEATURE FUNCTIONS
    # =========================
    feature_engine,

    # =========================
    # SETTINGS
    # =========================
    knockout=False,
    et_total=0.8
    ):

    """
    Simulate a football match using trained models to get probability of goals, cards, corners,
    then use poisson from numpy to get the exact values for these data

    In knockout mode, 
        - Average goals extra time in the WC ~0.8 goals per match, compute the expected goals
        - If draw after extra time, penalty will be True

    Returns:
        if knockout is False, return values will be a dict
        {  
            goals_home, goals_away, 
            yellow_home, yellow_away,
            red_home, red_away,
            corner_home, corner_away,
            result_str
        } 
        else, return values will be a dict
        {   
            goals_home, goals_away, 
            yellow_home, yellow_away,
            red_home, red_away,
            corner_home, corner_away,
            isPenalty,
            result_str
        }
        *result_str: "W" (A wins), "L" (B wins), "D" (draw)
    """
    
    goal_corner_cols = ["home_elo", "away_elo", "elo_diff", "home_advantage", "home_attack_rate", "home_defense_rate", "away_attack_rate", "away_defense_rate"]
    card_cols = goal_corner_cols + ["home_disc", "away_disc", "disc_diff", "disc_sum"]

    # =========================
    # FEATURE ENGINEERING
    # =========================
    match_row = feature_engine["combine_elo"](match_row, elo_ratings) # "home_elo", "away_elo", "elo_diff"

    match_row["home_advantage"] = int(feature_engine["home_adv"](match_row["home_team"], match_row["venue"]))
    
    match_row = feature_engine["match_features"](match_row, team_hist) # "home_attack_rate", "home_defense_rate", "away_attack_rate", "away_defense_rate", "home_disc", "away_disc", "disc_diff", "disc_sum"
    
    # Reshape
    X_goals = match_row[goal_corner_cols].values.reshape(1, -1)
    X_cards = match_row[card_cols].values.reshape(1, -1)


    # =========================
    # PREDICT LAMBDAS
    # =========================
    lam_home_goal = models["goal_home"].predict(X_goals)[0]
    lam_away_goal = models["goal_away"].predict(X_goals)[0]

    
    lam_home_yellow = models["yellow_home"].predict(X_cards)[0]
    lam_away_yellow = models["yellow_away"].predict(X_cards)[0]
    
    lam_home_red = models["red_home"].predict(X_cards)[0]
    lam_away_red = models["red_away"].predict(X_cards)[0]


    lam_home_corner = models["corner_home"].predict(X_goals)[0]
    lam_away_corner = models["corner_away"].predict(X_goals)[0]


    # =========================
    # 90 MIN SIMULATION
    # =========================
    home_goals = np.random.poisson(max(0, lam_home_goal))
    away_goals = np.random.poisson(max(0, lam_away_goal))

    home_yellow = np.random.poisson(max(0, lam_home_yellow))
    away_yellow = np.random.poisson(max(0, lam_away_yellow))

    home_corners = np.random.poisson(max(0, lam_home_corner))
    away_corners = np.random.poisson(max(0, lam_away_corner))

    home_red = int(np.random.random() < max(0, lam_home_red))
    away_red = int(np.random.random() < max(0, lam_away_red))

    
    # =========================
    # EXTRA TIME (KO ONLY)
    # =========================
    isPenalty = False
    penalty_winner = None

    if knockout and home_goals == away_goals:

        total = lam_home_goal + lam_away_goal
        if total <= 0:
            h_share = 0.5
            a_share = 0.5
        else:
            h_share = lam_home_goal / total
            a_share = lam_away_goal / total

        lam_home_et = et_total * h_share
        lam_away_et = et_total * a_share

        et_home = np.random.poisson(lam_home_et)
        et_away = np.random.poisson(lam_away_et)

        home_goals += et_home
        away_goals += et_away

        penalty_winner = None

        if et_home == et_away:
            isPenalty = True
            if np.random.random() < 0.5:
                penalty_winner = match_row["home_team"].iloc[0]
            else:
                penalty_winner = match_row["away_team"].iloc[0]

    # =========================
    # RESULT
    # =========================
    if home_goals > away_goals:
        result_str = "W"
    elif home_goals < away_goals:
        result_str = "L"
    else:
        result_str = "D"

    if isPenalty:
        if penalty_winner == match_row["home_team"].iloc[0]:
            result_str = "W"
        elif penalty_winner == match_row["away_team"].iloc[0]:
            result_str = "L"

    result = {
        "home_goals": home_goals,
        "away_goals": away_goals,
        "home_yellow": home_yellow,
        "away_yellow": away_yellow,
        "home_red": home_red,
        "away_red": away_red,
        "home_corners": home_corners,
        "away_corners": away_corners,
        "penalty": isPenalty,
        "result_str": result_str
    }

    return result

In [49]:
def simulate_group_stage(stage_table_df, simulate_match_fn, models, elo_ratings, team_hist, feature_engine, verbose = False):
    """
    Simulate group stage from a match schedule dataframe.
    """

    # =========================
    # PREP DATA
    # =========================
    stage_table_df = stage_table_df.copy()
    stage_table_df["date"] = pd.to_datetime(stage_table_df["date_utc"], utc=True)

    groups = stage_table_df["group"].unique()

    # =========================
    # PRECOMPUTE GROUP DATA
    # =========================
    group_matches_map = {
        g: stage_table_df[stage_table_df["group"] == g]
        for g in groups
    }

    group_teams_map = {
        g: pd.unique(df[["home_team", "away_team"]].values.ravel())
        for g, df in group_matches_map.items()
    }

    # =========================
    # INIT STATS
    # =========================
    group_stats = {}

    for g in groups:
        group_stats[g] = {
            t: {
                "pts": 0, "gf": 0, "ga": 0, "gd": 0,
                "w": 0, "d": 0, "l": 0,
                "yc": 0, "rc": 0, "corners": 0
            }
            for t in group_teams_map[g]
        }

    
    # store results for replay / debugging
    results = []

    # =========================
    # SIMULATE EACH MATCH
    # =========================
    for idx, row in stage_table_df.iterrows():

        group = row.group
        stats = group_stats[group]
        group_teams = group_teams_map[group] 

        match_row = pd.DataFrame({
            "home_team": [row.home_team],
            "away_team": [row.away_team],
            "date": [row.date],
            "venue": [row.venue]
        })
        
        result = simulate_match_fn(match_row, models, elo_ratings, team_hist, feature_engine)
        
        home = row.home_team
        away = row.away_team

        home_goals = result["home_goals"]
        away_goals = result["away_goals"]
        home_yellow = result["home_yellow"]
        away_yellow = result["away_yellow"]
        home_red = result["home_red"]
        away_red = result["away_red"]
        home_corners = result["home_corners"]
        away_corners = result["away_corners"]
        result_str = result["result_str"]
        

        # =========================
        # UPDATE STATS
        # =========================

        # goals for / against
        stats[home]["gf"] += home_goals
        stats[home]["ga"] += away_goals
        

        stats[away]["gf"] += away_goals
        stats[away]["ga"] += home_goals

        # yellow cards
        stats[home]["yc"] += home_yellow
        stats[away]["yc"] += away_yellow

        # red cards
        stats[home]["rc"] += home_red
        stats[away]["rc"] += away_red

        # corners
        stats[home]["corners"] += home_corners
        stats[away]["corners"] += away_corners

        winning_team = ""
        # result
        if result_str == "W":
            stats[home]["pts"] += 3
            stats[home]["w"] += 1
            stats[away]["l"] += 1
            winning_team = "home"
        elif result_str == "L":
            stats[away]["pts"] += 3
            stats[away]["w"] += 1
            stats[home]["l"] += 1
            winning_team = "away"
        else:
            stats[home]["pts"] += 1
            stats[away]["pts"] += 1
            stats[home]["d"] += 1
            stats[away]["d"] += 1
            winning_team = "draw"
                
        # =========================
        # STORE RESULT
        # =========================
        result_record = {
            "match_id": row.match_id,
            "group": group,
            "home_team": home,
            "away_team": away,
            "predicted_home_goals": home_goals,
            "predicted_away_goals": away_goals,
            "yellow_cards": home_yellow + away_yellow,
            "red_cards": home_red + away_red,
            "corners": home_corners + away_corners,
            "winning_team": winning_team # home, away, or draw
        }
        results.append(result_record)

        # =========================
        # GOAL DIFFERENCE
        # =========================
        stats[home]["gd"] = stats[home]["gf"] - stats[home]["ga"]
        stats[away]["gd"] = stats[away]["gf"] - stats[away]["ga"]

        # =========================
        # SORT TEAMS - FIFA tiebreakers: Points → Goal Difference → Goals Scored
        # =========================
        ranking = sorted(
            group_teams,
            key=lambda x: (
                stats[x]["pts"],
                stats[x]["gd"],
                stats[x]["gf"]
            ),
            reverse=True # sort in descending order
        )

        # =========================
        # PRINT STATS
        # =========================
        if verbose:
            print(f"\n{'═' * 60}")
            print(f"  GROUP {group}")
            print(f"{'═' * 60}\n")

            print(f"  {'#':<3} {'Team':<25} {'Pts':<4} {'W':<4} {'D':<4} {'L':<4} {'GF':<4} {'GA':<4} {'GD':<4} {'Corners':<4} {'YC':<4} {'RC':<4}")
            print(f"{'-' * 60}")

            for i, t in enumerate(ranking):
                s = stats[t]

                print(f"{i+1:<3} {t:<25} {s['pts']:>4} {s['w']:>4} "
                    f"{s['d']:>4} {s['l']:>4} {s['gf']:>4} {s['ga']:>4} "
                    f"{s['gd']:>4} {s['corners']:>7} {s['yc']:>4} "
                    f"{s['rc']:>4}")

            print(f"\nMatch Results - Group {group}:")
            for r in results:
                if r["group"] == group:
                    print(f"  {r['home_team']:<15} {r['predicted_home_goals']} - {r['predicted_away_goals']} {r['away_team']}")
            
    return {
        "group_stats": group_stats,
        "group_results": results,
        "stage_table_df": stage_table_df
    }

In [50]:
def get_round_of_32(group_stats, verbose = False):
    """
    Top 2 from each group (24) + 8 best 3rd-placed teams = 32.
    Tiebreakers for 3rd place: Pts → GD → GF.
    """

    # =========================
    # PREP DATA
    # =========================
    group_stats = copy.deepcopy(group_stats)

    qualifiers = []
    third_pool = []

    # =========================
    # PROCESS EACH GROUP
    # =========================
    for gname, teams_stats in group_stats.items():

        # FINAL RANKING
        ranking = sorted(
            teams_stats.keys(),
            key=lambda x: (
                teams_stats[x]["pts"],
                teams_stats[x]["gd"],
                teams_stats[x]["gf"],
                -teams_stats[x]["yc"], # - to sort in descending order
                -teams_stats[x]["rc"]
            ),
            reverse=True
        )

        # =========================
        # TOP 2 QUALIFY
        # =========================
        # top 1
        qualifiers.append(
            {
                "team": ranking[0],
                "group": gname,
                "pos":  1
            }
        )

        # top 2
        qualifiers.append(
            {
                "team": ranking[1],
                "group": gname,
                "pos":  2
            }
        )

        # =========================
        # 3RD PLACED TEAMS
        # =========================
        if len(ranking) >= 3:
            t = ranking[2]
            s = teams_stats[t]

            third_pool.append({
                "team": t,
                "group": gname,
                "pts": s["pts"],
                "gd": s["gd"],
                "gf": s["gf"],
                "yc": s["yc"],
                "rc": s["rc"],
                "pos": 3
            })

    # =========================
    # SORT 3RD POOL
    # =========================
    third_pool.sort(
        key=lambda x: (
            x["pts"],
            x["gd"],
            x["gf"],
            -x["yc"],
            -x["rc"]
        ),
        reverse=True
    )

    # =========================
    # GET BEST 8 3RD PLACED TEAMS
    # =========================
    best_third = third_pool[:8]
    r32 = qualifiers + [
            {
                "team": t["team"],
                "group": t["group"],
                "pos": t["pos"]
            }
            for t in best_third
        ]

    if verbose:
        eliminated = third_pool[8:]
        print(f"\n{'═' * 60}")
        print(f"  ROUND OF 32 - QUALIFICATION SUMMARY")
        print(f"{'═' * 60}\n")
        print(f"\n  🟢 Best 3rd-Place Teams (advance to R32):")
        for i, t in enumerate(best_third):
            print(f"  {i+1:<2}. {t['team']:<16} {t['group']} {t['pts']:>3} {t['gd']:>3} {t['gf']:>3} {t['yc']:>3} {t['rc']:>3}")
        print(f"\n  🔴 Eliminated Teams (remain in 3rd Pool):")
        for i, t in enumerate(eliminated):
            print(f"  {i+1:<2}. {t['team']:<16} {t['group']} {t['pts']:>3} {t['gd']:>3} {t['gf']:>3} {t['yc']:>3} {t['rc']:>3}")
        print(f"\n{'═' * 60}")
        print(f"\n  📊 Total Qualifiers: {len(qualifiers)}")

    return {
        "round_name": "Round of 32",
        "qualifiers": qualifiers,
        "third_pool": third_pool,
        "r32": r32
    }

In [51]:
def knockout_map(previous_round_results):
    """
    This is used to map knockout brackets from Q16 -> final
    """
    winner_map = {}
    loser_map = {}

    for match in previous_round_results["knockout_results"]:

        match_id = match["match_id"]

        winner = match["match_winner"]

        # ---------------------------------------------
        # Determine loser
        # ---------------------------------------------
        loser = (
            match["predicted_away_team"] if winner == match["predicted_home_team"]
            else match["predicted_home_team"]
        )

        winner_map[match_id] = winner
        loser_map[match_id] = loser

    return winner_map, loser_map
    

In [52]:
def fill_knockout_table(knockout_df, round_name, qualifiers, previous_round_results=None):
    """
    Populate knockout slots with actual teams.

    Parameters
    -------------------------------
    knockout_df : DataFrame
        Full knockout bracket dataframe.

    round_name : str
        Current round name.

    qualifiers : list[dict]
        Qualifiers of group stage

    previous_round_results : dict
        Results dict from previous knockout round.
        Must contain in "results" key:
            - match_id
            - match_winner
            - predicted_home_team
            - predicted_away_team
    """

    # =====================================================
    # FILTER ROUND
    # =====================================================
    df_round = knockout_df[
            knockout_df["round"] == round_name
        ].copy()

    # =====================================================
    # ROUND OF 32
    # =====================================================

    if round_name == "Round of 32":
        # =========================
        # BUILD LOOKUPS
        # =========================
        group_winners = {}
        group_runners = {}
        group_thirds = []

        for qualifier in qualifiers:
            if qualifier["pos"] == 1:
                group_winners[qualifier["group"]] = qualifier["team"]
            elif qualifier["pos"] == 2:
                group_runners[qualifier["group"]] = qualifier["team"]
            elif qualifier["pos"] == 3:
                group_thirds.append(qualifier["team"])

        third_index = 0

        # =========================
        # SLOT RESOLVER
        # =========================
        def resolve(slot):
            nonlocal third_index

            # Winner Group X
            if "Winner Group" in slot:
                group = slot.split()[-1]
                return group_winners.get(group)

            # Runner-up Group X
            if "Runner-up Group" in slot:
                group = slot.split()[-1]
                return group_runners.get(group)

            # Best 3rd
            if "Best 3rd" in slot:
                if third_index >= len(group_thirds):
                    return None
                team = group_thirds[third_index]
                third_index += 1
                return team

            return None  # strict fallback (prevents silent bad data)

    # =====================================================
    # ALL OTHER ROUNDS (R16, R8, R4, Final, Third place)
    # =====================================================
    else:
        winner_map, loser_map = knockout_map(previous_round_results)
        
        def resolve(slot):
            
            if slot is None:
                return None

            if not isinstance(slot, str):
                return slot

            # ---------------------------------------------
            # Winner Match X
            # ---------------------------------------------
            if "Winner Match" in slot:

                match_id = int(slot.split()[-1])

                return winner_map.get(match_id)

            # ---------------------------------------------
            # Loser Match X
            # ---------------------------------------------
            if "Loser Match" in slot:

                match_id = int(slot.split()[-1])

                return loser_map.get(match_id)
        
            return slot

    # =====================================================
    # FILL TEAMS
    # =====================================================
    knockout_df = knockout_df.copy()

    df_round["predicted_home_team"] = df_round["slot_home"].apply(resolve)
    df_round["predicted_away_team"] = df_round["slot_away"].apply(resolve) 
    
    # =====================================================
    # WRITE BACK TO FULL DF (CRITICAL)
    # =====================================================
    knockout_df.loc[df_round.index, "predicted_home_team"] = df_round["predicted_home_team"].values
    knockout_df.loc[df_round.index, "predicted_away_team"] = df_round["predicted_away_team"].values
        
    return knockout_df

In [ ]:
def knockout_simulate(round_name, qualifiers_data, knockout_df, simulate_match_fn, models, elo_ratings, team_hist, feature_engine, previous_round_results = None, verbose = False):
    """
    Simulate a knockout round (Round of 32, 16, QF, SF, Final).
    """
    if verbose:
        print(f"\n{'═' * 60}")
        print(f"  {round_name}")
        print(f"{'═' * 60}\n")
        print(f"  {'#':<4} {'Home':<16} {'':>2}{'Score':>7}{'':>2} "
              f"{'Away':<16}  {'Winner'}")
        print(f"  {'─' * 60}")
    knockout_df = knockout_df.copy()


    knockout_df["date"] = pd.to_datetime(knockout_df["date_utc"], utc=True)

    # =====================================================
    # FILTER MATCHES
    # =====================================================
    matches = knockout_df[knockout_df["round"] == round_name].copy()

    if round_name == "Round of 32":
        matches = fill_knockout_table(
            matches,
            round_name,
            qualifiers_data["r32"],
            None
        )
    else:
        winner_map, loser_map = knockout_map(previous_round_results)

        def resolve(slot):
            if slot is None:
                return None

            if not isinstance(slot, str):
                return slot

            # ---------------------------------------------
            # Winner Match X
            # ---------------------------------------------
            if "Winner Match" in slot:

                match_id = int(slot.split()[-1])

                return winner_map.get(match_id)

            # ---------------------------------------------
            # Loser Match X
            # ---------------------------------------------
            if "Loser Match" in slot:

                match_id = int(slot.split()[-1])

                return loser_map.get(match_id)
        
            return slot

        # fill matching teams
        matches["predicted_home_team"] = matches["slot_home"].apply(resolve)
        matches["predicted_away_team"] = matches["slot_away"].apply(resolve)
    

    results = []

    # =========================
    # SIMULATE EACH MATCH
    # =========================
    for i, row in matches.iterrows():
        home, away = row["predicted_home_team"], row["predicted_away_team"]
        date = row["date"]

        match_row = pd.DataFrame({
            "home_team": [home],
            "away_team": [away],
            "date": [date],
            "venue": [row.venue]
        })

        result = simulate_match_fn(
            match_row,
            models,
            elo_ratings,
            team_hist,
            feature_engine,
            knockout=True,
            et_total=0.8
        )

        home_goals = result["home_goals"]
        away_goals = result["away_goals"]
        home_yellow = result["home_yellow"]
        away_yellow = result["away_yellow"]
        home_red = result["home_red"]
        away_red = result["away_red"]
        home_corners = result["home_corners"]
        away_corners = result["away_corners"]
        isPenalty = result["penalty"]
        result_str = result["result_str"]
        
        idx = row.name

        match_winner = None
        match_loser = None
        winning_team = ""
        if result_str == "W":
            match_winner = home
            match_loser = away
            winning_team = "home"
        elif result_str == "L":
            match_winner = away
            match_loser = home
            winning_team = "away"


        results.append({
            "match_id": row["match_id"],
            "round": round_name,
            "predicted_home_team": home,
            "predicted_away_team": away,
            "predicted_home_goals": home_goals,
            "predicted_away_goals": away_goals,
            "yellow_cards": home_yellow + away_yellow,
            "red_cards": home_red + away_red,
            "corners": home_corners + away_corners,
            "penalties": isPenalty, # True or False
            "match_winner": match_winner, # Team name will be stored (e.g: Brazil)
            "match_loser": match_loser, # Team name will be stored (e.g: Brazil)
            "winning_team": winning_team # home or away
        })

        if verbose:
            print(
                f"  {row['match_id']:<4} "
                f"{home:<16} {home_goals}-{away_goals:<5} "
                f"{away:<16} => {match_winner}"
            )

    return {
        "round_name": round_name,
        "knockout_results": results,
        "knockout_df": knockout_df
    }

In [54]:
def world_cup_simulation(
    stage_table_df,
    knockout_df,
    simulate_match_fn,
    models,
    elo_ratings,
    team_hist,
    feature_engine,
    verbose = False, 
    seed = None
    ):
    """ Run a complete FIFA World Cup 2026 simulation. """
    if seed is not None:
        random.seed()
        np.random.seed(seed)

    if verbose:
        banner = "  🏆  FIFA WORLD CUP 2026 — FULL SIMULATION  🏆"
        print(f"\n{'═' * 60}")
        print(banner)
        print(f"  Hosted by USA · CANADA · MEXICO")
        print(f"  48 Teams  |  12 Groups  |  R32 → Final")
        print(f"{'═' * 60}")

    # ── Group Stage ──
    # Run simulate_group_stage
    group_stage_return = simulate_group_stage(
        stage_table_df, 
        simulate_match_fn, 
        models, 
        elo_ratings,
        team_hist,
        feature_engine,
        verbose
        )
    group_stats = group_stage_return["group_stats"]
    
    if verbose:
        print("##################### DONE WITH GROUP STAGE SIMULATION #####################")

    # Run get_round_of_32
    get_round_of_32_return = get_round_of_32(
        group_stats,
        verbose
    )
    
    # Run knockout_simulate
    # ==================================
    # ROUND OF 32
    # ==================================
    r32_return = knockout_simulate(
        "Round of 32",
        get_round_of_32_return,
        knockout_df,
        simulate_match_fn,
        models,
        elo_ratings,
        team_hist,
        feature_engine,
        previous_round_results=None,
        verbose=verbose
    )
    if verbose:
     print("##################### DONE WITH ROUND OF 32 SIMULATION #####################")
    # ==================================
    # ROUND OF 16
    # ==================================
    r16_return = knockout_simulate(
        "Round of 16",
        get_round_of_32_return,
        r32_return["knockout_df"],
        simulate_match_fn,
        models,
        elo_ratings,
        team_hist,
        feature_engine,
        previous_round_results=r32_return,
        verbose=verbose
    )
    if verbose:
        print("##################### DONE WITH ROUND OF 16 SIMULATION #####################")
    # ==================================
    # QUARTER-FINAL
    # ==================================
    qf_return = knockout_simulate(
        "Quarter-final",
        get_round_of_32_return,
        r16_return["knockout_df"],
        simulate_match_fn,
        models,
        elo_ratings,
        team_hist,
        feature_engine,
        previous_round_results=r16_return,
        verbose=verbose
    )

    if verbose:
        print("##################### DONE WITH QUARTER-FINAL SIMULATION #####################")

    # ==================================
    # SEMI-FINAL
    # ==================================
    sf_return = knockout_simulate(
        "Semi-final",
        get_round_of_32_return,
        qf_return["knockout_df"],
        simulate_match_fn,
        models,
        elo_ratings,
        team_hist,
        feature_engine,
        previous_round_results=qf_return,
        verbose=verbose
    )

    if verbose:
        print("##################### DONE WITH SEMI-FINAL SIMULATION #####################")

    # ==================================
    # FINAL
    # ==================================
    f_return = knockout_simulate(
        "Final",
        get_round_of_32_return,
        sf_return["knockout_df"],
        simulate_match_fn,
        models,
        elo_ratings,
        team_hist,
        feature_engine,
        previous_round_results=sf_return,
        verbose=verbose
    )

    if verbose:
        print("##################### DONE WITH FINAL SIMULATION #####################")

    final_home_team = f_return["knockout_results"][0]["predicted_home_team"]
    final_away_team = f_return["knockout_results"][0]["predicted_away_team"]
    final_home_score = f_return["knockout_results"][0]["predicted_home_goals"]
    final_away_score = f_return["knockout_results"][0]["predicted_away_goals"]
    champion = f_return["knockout_results"][0]["match_winner"]
    runner_up = f_return["knockout_results"][0]["match_loser"]

    # ==================================
    # THIRD PLACE
    # ==================================
    third_place_return = knockout_simulate(
        "Third-place playoff",
        get_round_of_32_return,
        sf_return["knockout_df"],
        simulate_match_fn,
        models,
        elo_ratings,
        team_hist,
        feature_engine,
        previous_round_results=sf_return,
        verbose=verbose
    )

    if verbose:
        print("##################### DONE WITH THIRD-PLACE PLAYOFF SIMULATION #####################")

    third = third_place_return["knockout_results"][0]["match_winner"]

    # FINAL RESULT - Combine R32, R16, QF, SF, Final and 3rd Place
    final_knockout_results = r32_return["knockout_results"] + r16_return["knockout_results"] + qf_return["knockout_results"] + sf_return["knockout_results"] + third_place_return["knockout_results"] + f_return["knockout_results"]


    if verbose:
        print(f"\n{'🏆' * 30}")
        print(f"                      FINAL")
        print(f"{'🏆' * 30}")
        print(f"  {final_home_team:<16} {final_home_score} - {final_away_score} {final_away_team:<16}")
        print(f"\n  🏆  WORLD CHAMPION :  {champion}")
        print(f"  🥈  RUNNER-UP      :  {runner_up}")
        print(f"  🥉  THIRD PLACE    :  {third}")
        print(f"{'🏆' * 30}\n")

    return {
        "champion": champion, 
        "runner_up": runner_up, 
        "third": third, 
        "group_stage_results": group_stage_return["group_results"],
        "knockout_results": final_knockout_results,
        }

In [55]:
# Load models

with open("models/goal_home.pkl", "rb") as f:
    goal_home_best = pickle.load(f)

with open("models/goal_away.pkl", "rb") as f:
    goal_away_best = pickle.load(f)

with open("models/yellow_home.pkl", "rb") as f:
    yellow_home_best = pickle.load(f)

with open("models/yellow_away.pkl", "rb") as f:
    yellow_away_best = pickle.load(f)

with open("models/red_home.pkl", "rb") as f:
    red_home_best = pickle.load(f)

with open("models/red_away.pkl", "rb") as f:
    red_away_best = pickle.load(f)
 
with open("models/corner_home.pkl", "rb") as f:
    corner_home_best = pickle.load(f)

with open("models/corner_away.pkl", "rb") as f:
    corner_away_best = pickle.load(f)


models = {
    "goal_home": goal_home_best,
    "goal_away": goal_away_best,
    "yellow_home": yellow_home_best,
    "yellow_away": yellow_away_best,
    "red_home": red_home_best,
    "red_away": red_away_best,
    "corner_home": corner_home_best,
    "corner_away": corner_away_best
}

In [56]:
# define feature engine
feature_engine = {
       "combine_elo": combine_teams_elo,
       "home_adv": is_home_advantage,
       "match_features": add_match_features
    }

In [57]:
gf = group_fixtures.copy()
gf["home_team"] = gf["home_team"].map(resolve_team_original_to_updated)
gf["away_team"] = gf["away_team"].map(resolve_team_original_to_updated)

elo_ratings = elo_ratings.copy()
elo_ratings["Team"] = elo_ratings["Team"].replace({"United States":"USA"})

In [58]:
world_cup_simulation(
    gf,
    knockout_slots,
    simulate_match,
    models,
    elo_ratings,
    team_hist,
    feature_engine,
    True,
    42
)


════════════════════════════════════════════════════════════
  🏆  FIFA WORLD CUP 2026 — FULL SIMULATION  🏆
  Hosted by USA · CANADA · MEXICO
  48 Teams  |  12 Groups  |  R32 → Final
════════════════════════════════════════════════════════════

════════════════════════════════════════════════════════════
  GROUP A
════════════════════════════════════════════════════════════

  #   Team                      Pts  W    D    L    GF   GA   GD   Corners YC   RC  
------------------------------------------------------------
1   Mexico                       3    1    0    0    4    0    4       5    4    1
2   South Korea                  0    0    0    0    0    0    0       0    0    0
3   Czechia                      0    0    0    0    0    0    0       0    0    0
4   South Africa                 0    0    0    1    0    4   -4       5    4    0

Match Results - Group A:
  Mexico          4 - 0 South Africa

════════════════════════════════════════════════════════════
  GROUP A
═════════

{'champion': 'Norway',
 'runner_up': 'Mexico',
 'third': 'Colombia',
 'group_stage_results': [{'match_id': 1,
   'group': 'A',
   'home_team': 'Mexico',
   'away_team': 'South Africa',
   'predicted_home_goals': 4,
   'predicted_away_goals': 0,
   'yellow_cards': 8,
   'red_cards': 1,
   'corners': 10,
   'winning_team': 'home'},
  {'match_id': 2,
   'group': 'A',
   'home_team': 'South Korea',
   'away_team': 'Czechia',
   'predicted_home_goals': 1,
   'predicted_away_goals': 0,
   'yellow_cards': 7,
   'red_cards': 0,
   'corners': 13,
   'winning_team': 'home'},
  {'match_id': 3,
   'group': 'B',
   'home_team': 'Canada',
   'away_team': 'Bosnia and Herzegovina',
   'predicted_home_goals': 1,
   'predicted_away_goals': 0,
   'yellow_cards': 8,
   'red_cards': 1,
   'corners': 6,
   'winning_team': 'home'},
  {'match_id': 4,
   'group': 'D',
   'home_team': 'USA',
   'away_team': 'Paraguay',
   'predicted_home_goals': 1,
   'predicted_away_goals': 0,
   'yellow_cards': 11,
   'red_ca

In [59]:
def monte_carlo(elo_ratings, original_group_df, original_knockout_df, n=1000, seed=42):
    """
    Run full Monte Carlo tournament simulation and return:
    - team performance table
    - final filled group dataframe
    - final filled knockout dataframe
    """
    # ===========================
    # DATAFRAME SET UP
    # ===========================
    gf = original_group_df.copy()
    gf["home_team"] = gf["home_team"].map(resolve_team_original_to_updated)
    gf["away_team"] = gf["away_team"].map(resolve_team_original_to_updated)

    elo_ratings = elo_ratings.copy()
    elo_ratings["Team"] = elo_ratings["Team"].replace({"United States":"USA"})



    champions, runners_up, thirds = [], [], []

    group_stage_sims = {}
    knockout_stage_sims = {}
    
    print("🔢 RUNNING MONTE CARLO SIMULATION 🔥")
    for i in range(n):
        print(f"\n\n Iteration: {i}")
        s = seed + i if seed is not None else None
        r = world_cup_simulation(
            gf,
            original_knockout_df,
            simulate_match,
            models,
            elo_ratings,
            team_hist,
            feature_engine,
            False,
            s
        )
        champions.append(r["champion"])
        runners_up.append(r["runner_up"])
        thirds.append(r["third"])
        
        group_stage_sims[i] = r["group_stage_results"]
        knockout_stage_sims[i] =r["knockout_results"]

    # =========================
    # TEAM FREQUENCIES
    # =========================
    c_cnt = pd.Series(champions).value_counts()
    r_cnt = pd.Series(runners_up).value_counts()
    t_cnt = pd.Series(thirds).value_counts()


    # =========================
    # MC MATCH AGGREGATION
    # =========================
    final_pred_dict = run_mc_pipeline(group_stage_sims, knockout_stage_sims)

    
    # =========================
    # FILL DATAFRAMES
    # =========================
    final_group_fixtures, final_knockout_slots = fill_predictions_df(original_group_df, original_knockout_df, final_pred_dict)


    rows = []
    # Convert from df to dict for easy lookup
    elo_ratings_dict = elo_ratings.set_index("Team")["Elo"].to_dict()
    for team in elo_ratings_dict.keys():
        rows.append({
            "Team":     team,
            "Rating":   elo_ratings_dict[team],
            "Champion": int(c_cnt.get(team, 0)),
            "Runner-Up": int(r_cnt.get(team, 0)),
            "3rd Place": int(t_cnt.get(team, 0)),
        })
    
    df = pd.DataFrame(rows)
    df["Podium"]  = df["Champion"] + df["Runner-Up"] + df["3rd Place"]
    df["Win %"]   = (df["Champion"] / n * 100).round(1)
    df["Pod %"]   = (df["Podium"]   / n * 100).round(1)
    df = df.sort_values(
        ["Champion", "Runner-Up", "3rd Place"], ascending=False
    ).reset_index(drop=True)
    df.index += 1
    return df, final_group_fixtures, final_knockout_slots

In [60]:
def display_monte_carlo(df, n):
    """Pretty-print Monte Carlo results."""
    sep = "  " + "─" * 68
    print(f"\n{'═' * 72}")
    print(f"  📊  MONTE CARLO ANALYSIS  —  {n:,} SIMULATIONS")
    print(f"{'═' * 72}")
    print(f"  {'#':<4} {'Team':<16} {'Rtg':>4} {'Wins':>5} {'RU':>4}"
          f" {'3rd':>4} {'Pod':>4} {'Win%':>7} {'Pod%':>7}")
    print(sep)

    shown = 0
    for idx, row in df.iterrows():
        if row["Champion"] > 0 or row["Podium"] >= 5:
            print(f"  {idx:<4} {row['Team']:<16} {row['Rating']:>4}"
                  f" {row['Champion']:>5} {row['Runner-Up']:>4}"
                  f" {row['3rd Place']:>4} {row['Podium']:>4}"
                  f" {row['Win %']:>6.1f}% {row['Pod %']:>6.1f}%")
            shown += 1
        if shown >= 25:
            break

    if shown < len(df):
        print(f"\n  ... and {len(df) - shown} more teams with 0 podium finishes.")

    top = df.iloc[0]
    print(f"\n  🏆  Most Likely Champion:  {top['Team']}"
          f"  ({top['Win %']}% win rate)")
    print(f"{'═' * 72}\n")

In [61]:
df, final_gf, final_ks = monte_carlo(elo_ratings.copy(), group_fixtures.copy(), knockout_slots.copy(), 2)

🔢 RUNNING MONTE CARLO SIMULATION 🔥


 Iteration: 0


 Iteration: 1


In [62]:
display_monte_carlo(df, 2)


════════════════════════════════════════════════════════════════════════
  📊  MONTE CARLO ANALYSIS  —  2 SIMULATIONS
════════════════════════════════════════════════════════════════════════
  #    Team              Rtg  Wins   RU  3rd  Pod    Win%    Pod%
  ────────────────────────────────────────────────────────────────────
  1    Spain            2165     1    0    0    1   50.0%   50.0%
  2    Norway           1912     1    0    0    1   50.0%   50.0%

  ... and 242 more teams with 0 podium finishes.

  🏆  Most Likely Champion:  Spain  (50.0% win rate)
════════════════════════════════════════════════════════════════════════



In [63]:
final_gf

,match_id,group,home_team,away_team,date_utc,venue,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,winning_team
0,1,A,Mexico,South Africa,2026-06-11T19:00:00Z,"Estadio Azteca, Mexico City",2.5,0.0,11.5,8.5,1.0,home
1,2,A,South Korea,Czechia,2026-06-12T02:00:00Z,"Estadio Akron, Guadalajara",3.5,0.0,12.5,9.0,0.0,home
2,3,B,Canada,Bosnia and Herzegovina,2026-06-12T19:00:00Z,"BMO Field, Toronto",1.0,0.5,4.5,9.0,1.0,home
3,4,D,USA,Paraguay,2026-06-13T01:00:00Z,"SoFi Stadium, Los Angeles",2.0,1.0,9.0,7.5,0.0,home
4,5,D,Australia,Turkey,2026-06-13T04:00:00Z,"BC Place, Vancouver",0.5,2.0,9.5,5.5,1.0,away
...,...,...,...,...,...,...,...,...,...,...,...,...
67,68,L,Croatia,Ghana,2026-06-27T21:00:00Z,"Lincoln Financial Field, Philadelphia",3.5,1.0,9.5,8.5,1.0,home
68,69,K,Colombia,Portugal,2026-06-27T23:30:00Z,"Hard Rock Stadium, Miami",1.0,2.0,9.0,9.5,0.0,home
69,70,K,DR Congo,Uzbekistan,2026-06-27T23:30:00Z,"Mercedes-Benz Stadium, Atlanta",0.0,1.0,10.0,5.5,0.0,away
70,71,J,Algeria,Austria,2026-06-28T02:00:00Z,"GEHA Field at Arrowhead Stadium, Kansas City",0.0,2.0,14.5,5.5,1.0,away


In [64]:
final_ks

,match_id,round,multiplier,date_utc,venue,slot_home,slot_away,predicted_home_team,predicted_away_team,predicted_home_goals,predicted_away_goals,corners,yellow_cards,red_cards,match_winner,penalties
0,73,Round of 32,1,2026-06-28T19:00:00Z,"SoFi Stadium, Los Angeles",Runner-up Group A,Runner-up Group B,South Korea,Switzerland,0.0,1.0,11.5,8.5,0.5,away,0.0
1,74,Round of 32,1,2026-06-29T17:00:00Z,"NRG Stadium, Houston",Winner Group C,Runner-up Group F,Brazil,Netherlands,1.0,0.5,7.0,8.5,1.0,away,0.0
2,75,Round of 32,1,2026-06-29T20:30:00Z,"Gillette Stadium, Boston",Winner Group E,Best 3rd (Groups A/B/C/D/F),Germany,Norway,1.0,2.5,10.5,5.5,0.5,away,0.0
3,76,Round of 32,1,2026-06-30T01:00:00Z,"Estadio BBVA, Monterrey",Winner Group F,Runner-up Group C,Japan,Scotland,0.0,2.0,11.5,9.0,0.5,away,0.0
4,77,Round of 32,1,2026-06-30T17:00:00Z,"AT&T Stadium, Dallas",Runner-up Group E,Runner-up Group I,Curaçao,France,2.5,2.0,14.5,7.5,0.0,home,0.5
5,78,Round of 32,1,2026-06-30T21:00:00Z,"MetLife Stadium, East Rutherford",Winner Group I,Best 3rd (Groups C/D/F/G/H),Senegal,Portugal,1.0,0.0,9.0,5.5,0.5,home,0.0
6,79,Round of 32,1,2026-07-01T01:00:00Z,"Estadio Azteca, Mexico City",Winner Group A,Best 3rd (Groups C/E/F/H/I),Mexico,Czechia,3.0,2.0,7.5,10.0,0.5,home,0.0
7,80,Round of 32,1,2026-07-01T16:00:00Z,"Mercedes-Benz Stadium, Atlanta",Winner Group L,Best 3rd (Groups E/H/I/J/K),Croatia,Paraguay,1.5,0.5,11.0,6.5,0.0,away,0.0
8,81,Round of 32,1,2026-07-01T20:00:00Z,"Lumen Field, Seattle",Winner Group G,Best 3rd (Groups A/E/H/I/J),Iran,Qatar,2.0,0.5,13.0,6.5,1.0,home,0.0
9,82,Round of 32,1,2026-07-02T00:00:00Z,"Levi's Stadium, Santa Clara",Winner Group D,Best 3rd (Groups B/E/F/I/J),USA,Uruguay,0.5,2.0,11.5,8.5,1.0,away,0.0
